<a href="https://colab.research.google.com/github/kaique-araujo/analysticalab/blob/main/mldexemplo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3

# Conecta ao banco de dados em memória
conn = sqlite3.connect(':memory:')
c = conn.cursor()

# 1. Cria a tabela de Clientes (com Chave Primária)
c.execute('''
    CREATE TABLE Clientes (
        cpf TEXT PRIMARY KEY,
        nome TEXT NOT NULL,
        email TEXT NOT NULL
    );
''')

# 2. Cria a tabela de Produtos (com Chave Primária)
c.execute('''
    CREATE TABLE Produtos (
        id_produto INTEGER PRIMARY KEY,
        nome_produto TEXT NOT NULL,
        preco REAL NOT NULL
    );
''')

# 3. Cria a tabela de Pedidos (com Chave Primária e Chave Estrangeira)
# Esta tabela se relaciona com a tabela de Clientes usando o CPF como Chave Estrangeira
c.execute('''
    CREATE TABLE Pedidos (
        id_pedido INTEGER PRIMARY KEY,
        data_pedido TEXT NOT NULL,
        cpf_cliente TEXT,
        FOREIGN KEY (cpf_cliente) REFERENCES Clientes(cpf)
    );
''')

# 4. Cria a Tabela Associativa para o relacionamento N-para-N entre Pedidos e Produtos
# Ela permite que um pedido tenha muitos produtos e um produto esteja em muitos pedidos.
c.execute('''
    CREATE TABLE ItensPedido (
        id_item INTEGER PRIMARY KEY,
        id_pedido INTEGER,
        id_produto INTEGER,
        quantidade INTEGER NOT NULL,
        FOREIGN KEY (id_pedido) REFERENCES Pedidos(id_pedido),
        FOREIGN KEY (id_produto) REFERENCES Produtos(id_produto)
    );
''')

# Inserindo dados de exemplo
c.execute("INSERT INTO Clientes VALUES ('111.222.333-44', 'João da Silva', 'joao@email.com')")
c.execute("INSERT INTO Clientes VALUES ('555.666.777-88', 'Maria Oliveira', 'maria@email.com')")

c.execute("INSERT INTO Produtos VALUES (101, 'Smartphone', 1500.00)")
c.execute("INSERT INTO Produtos VALUES (102, 'Notebook', 3500.00)")
c.execute("INSERT INTO Produtos VALUES (103, 'Fone de Ouvido', 250.00)")

c.execute("INSERT INTO Pedidos VALUES (1, '2023-10-26', '111.222.333-44')")
c.execute("INSERT INTO Pedidos VALUES (2, '2023-10-27', '555.666.777-88')")

# Inserindo dados na tabela associativa
c.execute("INSERT INTO ItensPedido VALUES (1, 1, 101, 1)") # Pedido 1 contém 1 Smartphone
c.execute("INSERT INTO ItensPedido VALUES (2, 1, 103, 2)") # Pedido 1 contém 2 Fones
c.execute("INSERT INTO ItensPedido VALUES (3, 2, 102, 1)") # Pedido 2 contém 1 Notebook

# Buscando dados para demonstrar a estrutura e as relações
print("--- Clientes ---")
for row in c.execute("SELECT * FROM Clientes;"):
    print(row)

print("\n--- Produtos ---")
for row in c.execute("SELECT * FROM Produtos;"):
    print(row)

print("\n--- Pedidos ---")
for row in c.execute("SELECT * FROM Pedidos;"):
    print(row)

print("\n--- Itens do Pedido (Tabela Associativa) ---")
for row in c.execute("SELECT * FROM ItensPedido;"):
    print(row)

# Exemplo de uma consulta que une tabelas (JOIN) para demonstrar as relações
print("\n--- Pedidos de João da Silva ---")
query = """
SELECT
    P.id_pedido, P.data_pedido, C.nome AS nome_cliente, Prod.nome_produto, IP.quantidade
FROM
    Pedidos AS P
JOIN
    Clientes AS C ON P.cpf_cliente = C.cpf
JOIN
    ItensPedido AS IP ON P.id_pedido = IP.id_pedido
JOIN
    Produtos AS Prod ON IP.id_produto = Prod.id_produto
WHERE
    C.nome = 'João da Silva';
"""
for row in c.execute(query):
    print(row)

# Commit e fecha a conexão
conn.commit()
conn.close()

--- Clientes ---
('111.222.333-44', 'João da Silva', 'joao@email.com')
('555.666.777-88', 'Maria Oliveira', 'maria@email.com')

--- Produtos ---
(101, 'Smartphone', 1500.0)
(102, 'Notebook', 3500.0)
(103, 'Fone de Ouvido', 250.0)

--- Pedidos ---
(1, '2023-10-26', '111.222.333-44')
(2, '2023-10-27', '555.666.777-88')

--- Itens do Pedido (Tabela Associativa) ---
(1, 1, 101, 1)
(2, 1, 103, 2)
(3, 2, 102, 1)

--- Pedidos de João da Silva ---
(1, '2023-10-26', 'João da Silva', 'Smartphone', 1)
(1, '2023-10-26', 'João da Silva', 'Fone de Ouvido', 2)


In [2]:
import datetime

# ==============================================================================
# 1. Definição das "Tabelas" (Classes Python)
#    Representa as entidades e seus atributos, conforme o modelo lógico.
# ==============================================================================

class Cliente:
    """
    Representa a tabela 'Cliente' no nosso modelo lógico de dados.
    Possui atributos como id_cliente (PK), nome, email, etc.
    """
    def __init__(self, id_cliente, nome_cliente, email_cliente, cpf_cliente, data_cadastro=None):
        # id_cliente é a Chave Primária (PK) - Identificador único de cada registro [4]
        self.id_cliente = id_cliente
        self.nome_cliente = nome_cliente # Atributo (coluna) [1, 3]
        self.email_cliente = email_cliente # Atributo (coluna)
        self.cpf_cliente = cpf_cliente   # Atributo (coluna)
        self.data_cadastro = data_cadastro if data_cadastro else datetime.date.today()

    def __repr__(self):
        return f"Cliente(ID={self.id_cliente}, Nome='{self.nome_cliente}', Email='{self.email_cliente}')"

class Pedido:
    """
    Representa a tabela 'Pedido' no nosso modelo lógico de dados.
    Possui atributos como id_pedido (PK), data, valor, status e id_cliente_fk (FK).
    """
    def __init__(self, id_pedido, data_pedido, valor_total, status_pedido, id_cliente_fk):
        # id_pedido é a Chave Primária (PK) - Identificador único de cada registro [4]
        self.id_pedido = id_pedido
        self.data_pedido = data_pedido     # Atributo (coluna) [1, 3]
        self.valor_total = valor_total     # Atributo (coluna)
        self.status_pedido = status_pedido # Atributo (coluna)
        # id_cliente_fk é a Chave Estrangeira (FK) - Conecta esta tabela à tabela Cliente [4]
        self.id_cliente_fk = id_cliente_fk

    def __repr__(self):
        return (f"Pedido(ID={self.id_pedido}, Data='{self.data_pedido}', "
                f"Valor={self.valor_total:.2f}, Cliente_ID_FK={self.id_cliente_fk})")

# ==============================================================================
# 2. Simulação de um "Banco de Dados" (listas de objetos)
#    No Colab, você usaria isso para testar a lógica antes de conectar a um SGBD real.
# ==============================================================================

# Lista para armazenar objetos Cliente (simulando a tabela CLIENTE)
db_clientes = []
# Lista para armazenar objetos Pedido (simulando a tabela PEDIDO)
db_pedidos = []

# ==============================================================================
# 3. Adicionar "Registros" (Instâncias de Classes)
#    Demonstra como as tuplas (linhas) seriam inseridas [1, 3].
# ==============================================================================

# Adicionar clientes (tuplas na tabela Cliente)
print("--- Adicionando Clientes ---")
cliente1 = Cliente(1, "Ana Silva", "ana.silva@email.com", "111.222.333-44")
cliente2 = Cliente(2, "Bruno Costa", "bruno.c@email.com", "555.666.777-88")
cliente3 = Cliente(3, "Carlos Oliveira", "carlos.o@email.com", "999.000.111-22")

db_clientes.append(cliente1)
db_clientes.append(cliente2)
db_clientes.append(cliente3)

for cliente in db_clientes:
    print(cliente)

print("\n--- Adicionando Pedidos ---")
# Adicionar pedidos (tuplas na tabela Pedido)
# Notar que id_cliente_fk aponta para um id_cliente existente [4]
pedido1 = Pedido(101, datetime.date(2023, 10, 26), 150.75, "Processando", cliente1.id_cliente) # Pedido da Ana (id_cliente = 1)
pedido2 = Pedido(102, datetime.date(2023, 10, 27), 300.00, "Enviado", cliente1.id_cliente)     # Outro pedido da Ana (id_cliente = 1)
pedido3 = Pedido(103, datetime.date(2023, 10, 27), 50.20, "Pendente", cliente2.id_cliente)     # Pedido do Bruno (id_cliente = 2)
pedido4 = Pedido(104, datetime.date(2023, 10, 28), 220.50, "Processando", cliente3.id_cliente) # Pedido do Carlos (id_cliente = 3)

db_pedidos.append(pedido1)
db_pedidos.append(pedido2)
db_pedidos.append(pedido3)
db_pedidos.append(pedido4)

for pedido in db_pedidos:
    print(pedido)

# Exemplo de integridade referencial (simulação):
# Tentar adicionar um pedido com um id_cliente_fk que não existe
# Em um SGBD real, isso resultaria em erro de integridade referencial [4]
print("\n--- Tentando adicionar Pedido com Cliente inexistente (simulação de erro) ---")
try:
    pedido_invalido = Pedido(105, datetime.date(2023, 10, 29), 99.99, "Novo", 999) # ID de cliente 999 não existe
    # db_pedidos.append(pedido_invalido) # Se descomentado, este seria um erro lógico
    print(f"Pedido com Cliente_ID_FK=999 não pode ser adicionado. (Em um SGBD, haveria erro de FK)")
except Exception as e:
    print(f"Erro simulado ao adicionar pedido: {e}")

# ==============================================================================
# 4. Consultar e Unir Dados (Simulando um JOIN)
#    Demonstra como o relacionamento 1:N entre Cliente e Pedido funciona [5].
# ==============================================================================

print("\n--- Consultando Pedidos e seus Clientes (Simulando JOIN) ---")
for pedido in db_pedidos:
    # Encontrar o cliente correspondente usando a Chave Estrangeira (id_cliente_fk)
    cliente_associado = next((c for c in db_clientes if c.id_cliente == pedido.id_cliente_fk), None)
    if cliente_associado:
        print(f"Pedido ID {pedido.id_pedido} (Valor: {pedido.valor_total:.2f}) feito por: {cliente_associado.nome_cliente}")
    else:
        print(f"Pedido ID {pedido.id_pedido} sem cliente associado (problema de integridade de dados!)")

print("\n--- Buscando todos os pedidos de um cliente específico (Ana Silva) ---")
ana_pedidos = [pedido for pedido in db_pedidos if pedido.id_cliente_fk == cliente1.id_cliente]
for pedido in ana_pedidos:
    print(pedido)

--- Adicionando Clientes ---
Cliente(ID=1, Nome='Ana Silva', Email='ana.silva@email.com')
Cliente(ID=2, Nome='Bruno Costa', Email='bruno.c@email.com')
Cliente(ID=3, Nome='Carlos Oliveira', Email='carlos.o@email.com')

--- Adicionando Pedidos ---
Pedido(ID=101, Data='2023-10-26', Valor=150.75, Cliente_ID_FK=1)
Pedido(ID=102, Data='2023-10-27', Valor=300.00, Cliente_ID_FK=1)
Pedido(ID=103, Data='2023-10-27', Valor=50.20, Cliente_ID_FK=2)
Pedido(ID=104, Data='2023-10-28', Valor=220.50, Cliente_ID_FK=3)

--- Tentando adicionar Pedido com Cliente inexistente (simulação de erro) ---
Pedido com Cliente_ID_FK=999 não pode ser adicionado. (Em um SGBD, haveria erro de FK)

--- Consultando Pedidos e seus Clientes (Simulando JOIN) ---
Pedido ID 101 (Valor: 150.75) feito por: Ana Silva
Pedido ID 102 (Valor: 300.00) feito por: Ana Silva
Pedido ID 103 (Valor: 50.20) feito por: Bruno Costa
Pedido ID 104 (Valor: 220.50) feito por: Carlos Oliveira

--- Buscando todos os pedidos de um cliente específico 